In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf master.zip

# 2. 克隆仓库
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    %cd Diffusion-Illusions
    print("正在安装依赖...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
else:
    print("❌❌ 克隆失败，请检查网络。")

In [ ]:
import os
repo_name = "Diffusion-Illusions"
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
from rp import *
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
from google.colab import files
from PIL import Image, ImageOps

# === 核心工具函数：模拟运动模糊 (Motion Blur) ===

def get_motion_blur_kernel(kernel_size, angle_deg, device):
    """
    生成一个模拟运动模糊的卷积核。
    kernel_size: 模糊的长度（摇晃幅度）
    angle_deg: 摇晃的角度（0度=左右摇晃，90度=上下摇晃）
    """
    # 创建一个空的核
    kernel_tensor = torch.zeros((kernel_size, kernel_size), device=device)
    center = kernel_size // 2

    # 在中间画一条线
    kernel_tensor[center, :] = 1.0

    # 旋转这条线
    # 注意：为了旋转，我们需要先扩充维度 [1, 1, H, W]
    kernel_tensor = kernel_tensor.unsqueeze(0).unsqueeze(0)
    kernel_tensor = TF.rotate(kernel_tensor, angle_deg)
    kernel_tensor = kernel_tensor.squeeze()

    # 归一化 (让亮度保持不变)
    kernel_tensor = kernel_tensor / kernel_tensor.sum()

    # 调整形状以适配 conv2d: [Out_Channels, In_Channels/Groups, H, W]
    # 我们是 RGB 3通道独立卷积，所以 Groups=3
    return kernel_tensor.view(1, 1, kernel_size, kernel_size).repeat(3, 1, 1, 1)

def apply_motion_blur(image, kernel):
    """
    对图片应用运动模糊
    image: [1, 3, H, W]
    kernel: 预先生成的模糊核
    """
    pad = kernel.shape[2] // 2
    # 使用 conv2d 模拟模糊
    return F.conv2d(image, kernel, padding=pad, groups=3)

In [ ]:
# 初始化 GPU 和 模型
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    print("模型加载完毕！")
else:
    print("模型已存在，跳过加载。")

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF

# === 5. 上传目标图片 (Target) ===
print(">>> 请点击下方按钮上传你的【玫瑰花】图片 <<<")
print("建议：最好是特写，颜色鲜艳，背景不要太杂")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))

    # 1. 读取图片
    target_pil = Image.open(filename).convert('RGB')

    # 2. 智能缩放：自动裁切并缩放到 256x256
    target_pil = ImageOps.fit(target_pil, (256, 256), method=Image.Resampling.LANCZOS)

    # 3. 转为 Tensor
    target_tensor = TF.to_tensor(target_pil).to(device).unsqueeze(0)

    print(f"\n✅ 成功加载玫瑰花: {filename}")
    print("目标预览 (摇晃后将显示此图)：")
    rp.display_image(rp.as_numpy_image(target_tensor.squeeze()))

else:
    print("❌ 未上传图片，请重新运行此代码块！")

In [ ]:
# === 6. 游戏参数与 Prompt 配置 (修正版) ===

# 【关键修改 1】降低模糊难度
# 改为 35 (必须是奇数)。这意味着更短促的摇晃就能看清，图像也更清晰。
BLUR_STRENGTH = 35       

# 保持 45 度对角线摇晃
SHAKE_ANGLE = 45         

# 【关键修改 2】暴力提升引导强度
# 从 5000 提到了 25000。这会强制像素对齐成玫瑰花，哪怕牺牲一点卡通楼房的自然度。
GUIDANCE_STRENGTH = 25000 

# === 🎨 关键：修改 Prompt 增加纹理 ===
# 纯扁平 (Flat) 风格很难藏东西。改为 "Crayon/Pencil" (蜡笔/铅笔) 风格，
# 利用笔触的噪点来隐藏玫瑰花的纹理，效果会好很多。
prompt_surface = "Children's crayon drawing of a city street, colorful buildings, red houses, green trees, textured paper, rough sketch, vibrant colors, messy cute style"

# 负面提示词
negative_prompt = "blur, smooth, photo, realistic, 3d render, dark, gloomy, low quality, flat color"

# === 初始化 ===
image_maker = lambda: LearnableImageFourier(height=256, width=256, hidden_dim=256, num_features=256).to(device)
raw_image = image_maker()

# 准备模糊核
blur_kernel = get_motion_blur_kernel(BLUR_STRENGTH, SHAKE_ANGLE, device)

# 标签
label_surface = NegativeLabel(prompt_surface, negative_prompt)

# 优化器
params = chain(raw_image.parameters())
optim = torch.optim.SGD(params, lr=1e-3)

print(f"配置完成。强度已拉满 (25000)，准备暴力隐藏玫瑰花！")

In [ ]:
# === 7. 开始训练 ===
NUM_ITER = 2500           
DISPLAY_INTERVAL = 200    

model_sd.max_step = 980
model_sd.min_step = 20

display_eta = rp.eta(NUM_ITER, title='Training Status')

print(f"🚀 开始训练... 表面是蜡笔画小镇，摇晃后必须是玫瑰！")

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)

        img = raw_image()

        # A. 表面 Loss (像蜡笔画)
        _ = model_sd.train_step(
            label_surface.embedding,
            img[None],
            noise_coef=0.1,
            guidance_scale=60
        )

        # B. 隐身 Loss (摇晃后像玫瑰)
        img_blurred = apply_motion_blur(img[None], blur_kernel)
        loss_secret = torch.mean((img_blurred - target_tensor)**2) * GUIDANCE_STRENGTH
        
        loss_secret.backward()

        # C. 显示
        with torch.no_grad():
            if iter_num % DISPLAY_INTERVAL == 0:
                from IPython.display import clear_output
                clear_output(wait=True)
                
                static_np = rp.as_numpy_image(img)
                shaken_np = rp.as_numpy_image(img_blurred.squeeze())
                target_np = rp.as_numpy_image(target_tensor.squeeze())

                print(f"Iteration {iter_num} / {NUM_ITER}")
                print(f"[左] 静止(蜡笔楼房)  |  [中] 摇晃(玫瑰花)  |  [右] 目标原图")
                
                combined = np.hstack([static_np, shaken_np, target_np])
                rp.display_image(combined)

        optim.step()
        optim.zero_grad()

except KeyboardInterrupt:
    print("用户手动停止训练。")

In [ ]:
import imageio
import numpy as np
from PIL import Image

# === 9. 终极高速抖动 GIF (High-Frequency Jitter) ===
print("==== 🎮 生成【高速抖动】版 GIF ====")

final_img_pil = TF.to_pil_image(raw_image().cpu())
frames = []

# === A. 参数调整 (暴力提速) ===
# GIF 的极限通常是 0.02s 一帧 (50fps)。我们利用这个极限。
fps = 50           
duration = 1.0     # 1秒就够了，循环播放
frames_count = int(fps * duration)

# 震动幅度 (像素)
amplitude = 17     
angle = 45         # 对角线

print(f"正在渲染 {frames_count} 帧 (极速震动模式)...")

for i in range(frames_count):
    # 关键修改：不再使用平滑的三角波，而是使用高频的 Cosine 波
    # i * 2.5 意味着每帧的变化极其剧烈，产生“重影”效果
    # 这种跳跃感能更好地模拟快速摇晃
    phase = np.cos(i * 2.5) 
    
    offset = phase * amplitude
    
    # 计算对角线分量
    import math
    rad = math.radians(angle)
    offset_x = int(offset * math.cos(rad))
    offset_y = int(offset * math.sin(rad))
    
    # 绘图
    im = final_img_pil.copy()
    im_size = im.size[0]
    frame_size = int(im_size * 1.5)
    frame = Image.new('RGB', (frame_size, frame_size), (255, 255, 255))
    
    paste_x = (frame_size - im_size) // 2 + offset_x
    paste_y = (frame_size - im_size) // 2 + offset_y
    frame.paste(im, (paste_x, paste_y))
    
    frames.append(np.array(frame))

# === B. 保存 GIF (关键设置) ===
save_path = "high_speed_shake.gif"

# duration=0.02 强制每帧间隔 20ms (即 50fps)
imageio.mimsave(save_path, frames, duration=0.02, loop=0)

print(f"✅ 极速动图已保存: {save_path}")
print("👇 现在的抖动应该非常鬼畜，眼睛无法聚焦在楼房上，从而看到玫瑰：")
rp.display_image(save_path)